step1_setup_schaefer400

In [ ]:
from pathlib import Path
from glob import glob
import json
import re

import nibabel as nib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from nilearn import image
from nilearn.plotting import plot_stat_map
from sklearn.linear_model import LinearRegression


# =========================
# 1. Project setup
# =========================

DATA_DIR = Path(r"CHANGE_THIS_TO_YOUR_BIDS_DATA_DIR")
PROJECT_DIR = Path(r"CHANGE_THIS_TO_YOUR_PROJECT_DIR")

# Keep a NEW run folder so the original pipeline remains untouched
RUN_NAME = f"{ROI_NAME}_400_parcels_cleaned"
ROI_NAME = "CHANGE_THIS_TO_YOUR_ROI_NAME"

RUN_DIR = PROJECT_DIR / "runs" / RUN_NAME
STEP_DIR = RUN_DIR / "step1_setup_roi_cleaned"
FIG_DIR = STEP_DIR / "figures"
CSV_DIR = STEP_DIR / "csv"
NPY_DIR = STEP_DIR / "npy"
JSON_DIR = STEP_DIR / "json"

for d in [PROJECT_DIR, RUN_DIR, STEP_DIR, FIG_DIR, CSV_DIR, NPY_DIR, JSON_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Data directory: {DATA_DIR}")
print(f"Project directory: {PROJECT_DIR}")
print(f"Run directory: {RUN_DIR}")
print(f"Step directory: {STEP_DIR}")


# =========================
# 2. Atlas setup
# =========================

ATLAS_PATH = Path(r"CHANGE_THIS_TO_YOUR_SCHAEFER400_ATLAS_NIFTI")

LABEL_TXT_PATH = Path(r"CHANGE_THIS_TO_YOUR_SCHAEFER2018_LABEL_TXT")


if not ATLAS_PATH.exists():
    raise FileNotFoundError(f"Atlas file not found: {ATLAS_PATH}")

atlas_nii = nib.load(str(ATLAS_PATH))
atlas_img = atlas_nii.get_fdata()

print(f"Loaded atlas from: {ATLAS_PATH}")
print(f"Atlas shape: {atlas_img.shape}")

# Optional labels
if LABEL_TXT_PATH.exists():
    label_rows = []
    with open(LABEL_TXT_PATH, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                label_rows.append({
                    "parcel_id": int(parts[0]),
                    "parcel_name": parts[1]
                })

    label_df = pd.DataFrame(label_rows)
    print(f"Loaded label table from official txt: {LABEL_TXT_PATH}")
else:
    label_df = None
    print("Label TXT not found. Proceeding without parcel names.")


# =========================
# 3. Selected Schaefer parcels
# =========================

SELECTED_PARCELS = [
    13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223]

print("\nSelected parcels:")
print(SELECTED_PARCELS)
print(f"Number of selected parcels: {len(SELECTED_PARCELS)}")

if label_df is not None:
    selected_label_df = label_df[label_df["parcel_id"].isin(SELECTED_PARCELS)].copy()
    selected_label_df = selected_label_df.sort_values("parcel_id")
    print("\nSelected parcel names:")
    print(selected_label_df.to_string(index=False))
    selected_label_df.to_csv(CSV_DIR / "selected_parcel_labels.csv", index=False, encoding="utf-8-sig")
else:
    selected_label_df = None


# =========================
# 4. Experiment metadata
# =========================

TR = 1.5
N_TOTAL_TR = 2249

# From dataset description:
# story starts at 0:21, TR = 1.5 -> start TR = 14
# story duration = 2226 TRs
STORY_START_TR = 14
STORY_N_TR = 2226
STORY_END_TR = STORY_START_TR + STORY_N_TR

print("\nExperiment metadata:")
print(f"TR = {TR}")
print(f"N_TOTAL_TR = {N_TOTAL_TR}")
print(f"STORY_START_TR = {STORY_START_TR}")
print(f"STORY_N_TR = {STORY_N_TR}")
print(f"STORY_END_TR = {STORY_END_TR}")

if STORY_END_TR > N_TOTAL_TR:
    raise ValueError("STORY_END_TR exceeds N_TOTAL_TR. Please re-check metadata.")


# =========================
# 5. Confound-cleaning settings
# =========================

BASE_CONFOUND_COLS = [
    "trans_x", "trans_y", "trans_z",
    "rot_x", "rot_y", "rot_z",
    "framewise_displacement",
    "dvars",
    "csf", "white_matter", "global_signal"
]

MAX_A_COMPCOR = 6
MAX_T_COMPCOR = 6

USE_COSINE_DRIFTS = True
APPLY_STANDARDIZE_Y = False   # keep false here to preserve original Step1 logic as much as possible

print("\nConfound settings:")
print(f"BASE_CONFOUND_COLS = {BASE_CONFOUND_COLS}")
print(f"MAX_A_COMPCOR = {MAX_A_COMPCOR}")
print(f"MAX_T_COMPCOR = {MAX_T_COMPCOR}")
print(f"USE_COSINE_DRIFTS = {USE_COSINE_DRIFTS}")
print(f"APPLY_STANDARDIZE_Y = {APPLY_STANDARDIZE_Y}")


# =========================
# 6. Discover BOLD files
# =========================

bold_fns = sorted(
    glob(str(DATA_DIR / "sub-*" / "func" / "*task-21styear*space-MNI152NLin6Asym_res-native_desc-preproc_bold.nii.gz"))
)

print(f"\nDiscovered preprocessed 21styear BOLD files: {len(bold_fns)}")


def parse_bold_filename(path_str: str) -> dict:
    path = Path(path_str)
    name = path.name

    subj_match = re.search(r"(sub-\d+)", name)
    task_match = re.search(r"(task-[^_]+)", name)

    subject = subj_match.group(1) if subj_match else None
    task = task_match.group(1) if task_match else None

    return {
        "subject": subject,
        "task": task,
        "path": str(path),
        "filename": name,
    }


def find_confounds_file(bold_path: Path) -> Path:
    """
    Match the confounds TSV corresponding to the BOLD file.
    """
    name = bold_path.name

    candidate1 = bold_path.with_name(
        name.replace(
            "_space-MNI152NLin6Asym_res-native_desc-preproc_bold.nii.gz",
            "_desc-confounds_regressors.tsv"
        )
    )
    candidate2 = bold_path.with_name(
        name.replace(
            "_space-MNI152NLin6Asym_res-native_desc-preproc_bold.nii.gz",
            "_desc-confounds_timeseries.tsv"
        )
    )

    if candidate1.exists():
        return candidate1
    if candidate2.exists():
        return candidate2

    folder = bold_path.parent
    subj_match = re.search(r"(sub-\d+)", name)
    task_match = re.search(r"(task-[^_]+)", name)

    subj = subj_match.group(1) if subj_match else None
    task = task_match.group(1) if task_match else None

    hits = sorted(folder.glob(f"{subj}_{task}_*confounds*.tsv"))
    if len(hits) > 0:
        return hits[0]

    raise FileNotFoundError(f"Could not find confounds TSV for: {bold_path}")


rows = [parse_bold_filename(fn) for fn in bold_fns]
df = pd.DataFrame(rows)
df = df.dropna(subset=["subject"]).copy()

# Keep exactly one valid file per subject
subject_rows = []
for subj in sorted(df["subject"].unique()):
    sub_df = df[df["subject"] == subj].copy()
    if len(sub_df) != 1:
        print(f"[WARNING] Subject {subj} has {len(sub_df)} matching files. Skipping.")
        continue

    selected = sub_df.iloc[0]
    bold_path = Path(selected["path"])

    try:
        confounds_path = find_confounds_file(bold_path)
    except Exception as e:
        print(f"[WARNING] Subject {subj}: confounds file not found. Skipping. Error: {e}")
        continue

    subject_rows.append({
        "subject": subj,
        "task": selected["task"],
        "bold_path": str(bold_path),
        "confounds_path": str(confounds_path),
    })

manifest_df = pd.DataFrame(subject_rows)
manifest_df.to_csv(CSV_DIR / "subject_file_manifest.csv", index=False, encoding="utf-8-sig")

print(f"\nSubjects retained after BOLD+confounds matching: {len(manifest_df)}")
print(manifest_df.head())


# =========================
# 7. Build Schaefer parcel mask
# =========================

roi_img = np.zeros(atlas_img.shape, dtype=np.uint8)
for parcel in SELECTED_PARCELS:
    roi_img[atlas_img == parcel] = 1

roi_nii = nib.Nifti1Image(roi_img, atlas_nii.affine, atlas_nii.header)
roi_mask_path = STEP_DIR / "selected_schaefer400_roi_mask.nii.gz"
nib.save(roi_nii, roi_mask_path)

print(f"\nSaved combined parcel ROI mask to: {roi_mask_path}")

roi_voxel_count_atlas = int(np.sum(roi_img > 0))
print(f"Atlas-space ROI voxel count: {roi_voxel_count_atlas}")


# =========================
# 8. Quick visualization of ROI mask
# =========================

sns.set(palette="colorblind")

plt.figure(figsize=(8, 6))
plot_stat_map(
    roi_nii,
    cmap="tab10_r",
    colorbar=False,
    title= f"Selected Schaefer-400 parcel set: {ROI_NAME}"
)
plt.savefig(FIG_DIR / "selected_schaefer400_roi.png", dpi=200, bbox_inches="tight")
plt.close()

print(f"Saved ROI figure to: {FIG_DIR / 'selected_schaefer400_roi.png'}")


# =========================
# 9. Confound helpers
# =========================

def zscore_cols(X: np.ndarray) -> np.ndarray:
    X = np.asarray(X, dtype=np.float64)
    mu = X.mean(axis=0, keepdims=True)
    sd = X.std(axis=0, keepdims=True, ddof=0)
    sd[sd < 1e-8] = 1.0
    return (X - mu) / sd


def safe_corr(x, y):
    x = np.asarray(x, dtype=np.float64).ravel()
    y = np.asarray(y, dtype=np.float64).ravel()

    if len(x) != len(y):
        return np.nan
    if np.std(x) < 1e-8 or np.std(y) < 1e-8:
        return np.nan

    return float(np.corrcoef(x, y)[0, 1])


def pick_confound_columns(conf_df: pd.DataFrame):
    cols = []

    for c in BASE_CONFOUND_COLS:
        if c in conf_df.columns:
            cols.append(c)

    a_cols = sorted([c for c in conf_df.columns if c.startswith("a_comp_cor")])[:MAX_A_COMPCOR]
    t_cols = sorted([c for c in conf_df.columns if c.startswith("t_comp_cor")])[:MAX_T_COMPCOR]

    cols.extend(a_cols)
    cols.extend(t_cols)

    if USE_COSINE_DRIFTS:
        cosine_cols = sorted([c for c in conf_df.columns if c.startswith("cosine")])
        cols.extend(cosine_cols)

    cols = list(dict.fromkeys(cols))
    return cols


def prepare_confounds(confounds_path: Path):
    conf_df = pd.read_csv(confounds_path, sep="\t")

    if len(conf_df) < STORY_END_TR:
        raise ValueError(
            f"Confounds rows shorter than required story window. "
            f"rows={len(conf_df)}, need at least {STORY_END_TR}"
        )

    conf_df_story = conf_df.iloc[STORY_START_TR:STORY_END_TR].reset_index(drop=True)

    use_cols = pick_confound_columns(conf_df_story)
    if len(use_cols) == 0:
        raise ValueError(f"No usable confound columns found in: {confounds_path}")

    Xc = conf_df_story[use_cols].copy()

    for c in Xc.columns:
        Xc[c] = pd.to_numeric(Xc[c], errors="coerce")

    Xc = Xc.bfill().ffill().fillna(0.0)

    Xc_np = Xc.values.astype(np.float64)
    Xc_np = zscore_cols(Xc_np)

    return conf_df_story, use_cols, Xc_np


def regress_out_confounds(Y: np.ndarray, Xc: np.ndarray):
    """
    Y: [time, voxels]
    Xc: [time, confounds]
    Returns:
      Y_clean
      Y_hat
      voxel_r2
    """
    Y = np.asarray(Y, dtype=np.float64)
    Xc = np.asarray(Xc, dtype=np.float64)

    model = LinearRegression(fit_intercept=True)
    model.fit(Xc, Y)
    Y_hat = model.predict(Xc)

    Y_clean = Y - Y_hat

    ss_res = np.sum((Y - Y_hat) ** 2, axis=0)
    ss_tot = np.sum((Y - Y.mean(axis=0, keepdims=True)) ** 2, axis=0)
    voxel_r2 = 1.0 - (ss_res / np.maximum(ss_tot, 1e-8))

    if APPLY_STANDARDIZE_Y:
        Y_clean = zscore_cols(Y_clean)

    return Y_clean.astype(np.float32), Y_hat.astype(np.float32), voxel_r2.astype(np.float32)


# =========================
# 10. Load functional data, extract raw ROI,
#     clean with confounds, and save per-subject diagnostics
# =========================

raw_data = []
cleaned_data = []

shape_rows = []
confound_rows = []
columns_rows = []

for _, row in manifest_df.iterrows():
    subj = row["subject"]
    task = row["task"]
    data_fn = Path(row["bold_path"])
    confounds_fn = Path(row["confounds_path"])

    print("\n" + "=" * 60)
    print(f"Processing {subj}")
    print("=" * 60)
    print(f"BOLD path: {data_fn}")
    print(f"Confounds path: {confounds_fn}")

    # -------------------------
    # A. Load BOLD and crop to story
    # -------------------------
    bold_nii = nib.load(str(data_fn))
    voxel_data = bold_nii.get_fdata(dtype=np.float32)

    if voxel_data.shape[-1] != N_TOTAL_TR:
        print(f"[WARNING] {subj}: expected {N_TOTAL_TR} TRs, got {voxel_data.shape[-1]} TRs")

    voxel_data = voxel_data[..., STORY_START_TR:STORY_END_TR]

    if voxel_data.shape[-1] != STORY_N_TR:
        raise ValueError(f"{subj}: cropped BOLD has {voxel_data.shape[-1]} TRs, expected {STORY_N_TR}")

    # -------------------------
    # B. Resample ROI mask to subject BOLD grid
    # -------------------------
    bold_ref_nii = nib.Nifti1Image(voxel_data[..., 0], bold_nii.affine, bold_nii.header)

    roi_resampled_nii = image.resample_to_img(
        roi_nii,
        bold_ref_nii,
        interpolation="nearest",
        force_resample=True,
        copy_header=True,
    )

    roi_resampled = roi_resampled_nii.get_fdata() > 0

    # Raw ROI data: [time, voxels]
    roi_data_raw = voxel_data[roi_resampled, :].T.astype(np.float32)

    if roi_data_raw.shape[0] != STORY_N_TR:
        raise ValueError(f"{subj}: raw ROI time dimension mismatch: {roi_data_raw.shape[0]} vs {STORY_N_TR}")

    # -------------------------
    # C. Load confounds and regress them out
    # -------------------------
    conf_df_story, used_confounds, Xc = prepare_confounds(confounds_fn)

    if Xc.shape[0] != roi_data_raw.shape[0]:
        raise ValueError(
            f"{subj}: time mismatch between ROI and confounds. "
            f"ROI={roi_data_raw.shape[0]} vs confounds={Xc.shape[0]}"
        )

    roi_data_clean, roi_data_hat, voxel_r2 = regress_out_confounds(roi_data_raw, Xc)

    # -------------------------
    # D. Subject-level diagnostics
    # -------------------------
    roi_mean_raw = roi_data_raw.mean(axis=1)
    roi_mean_clean = roi_data_clean.mean(axis=1)

    fd = conf_df_story["framewise_displacement"].values if "framewise_displacement" in conf_df_story.columns else None
    dvars = conf_df_story["dvars"].values if "dvars" in conf_df_story.columns else None
    gsig = conf_df_story["global_signal"].values if "global_signal" in conf_df_story.columns else None

    corr_fd_before = safe_corr(roi_mean_raw, fd) if fd is not None else np.nan
    corr_fd_after = safe_corr(roi_mean_clean, fd) if fd is not None else np.nan

    corr_dvars_before = safe_corr(roi_mean_raw, dvars) if dvars is not None else np.nan
    corr_dvars_after = safe_corr(roi_mean_clean, dvars) if dvars is not None else np.nan

    corr_gsig_before = safe_corr(roi_mean_raw, gsig) if gsig is not None else np.nan
    corr_gsig_after = safe_corr(roi_mean_clean, gsig) if gsig is not None else np.nan

    raw_data.append(roi_data_raw)
    cleaned_data.append(roi_data_clean)

    shape_rows.append({
        "subject": subj,
        "task": task,
        "n_timepoints": int(roi_data_raw.shape[0]),
        "n_voxels": int(roi_data_raw.shape[1]),
    })

    confound_rows.append({
        "subject": subj,
        "n_timepoints": int(roi_data_raw.shape[0]),
        "n_voxels": int(roi_data_raw.shape[1]),
        "n_confounds_used": int(Xc.shape[1]),
        "mean_voxel_r2_explained_by_confounds": float(np.mean(voxel_r2)),
        "median_voxel_r2_explained_by_confounds": float(np.median(voxel_r2)),
        "p90_voxel_r2_explained_by_confounds": float(np.percentile(voxel_r2, 90)),
        "positive_fraction_voxel_r2": float(np.mean(voxel_r2 > 0)),
        "corr_roi_mean_vs_fd_before": corr_fd_before,
        "corr_roi_mean_vs_fd_after": corr_fd_after,
        "corr_roi_mean_vs_dvars_before": corr_dvars_before,
        "corr_roi_mean_vs_dvars_after": corr_dvars_after,
        "corr_roi_mean_vs_global_before": corr_gsig_before,
        "corr_roi_mean_vs_global_after": corr_gsig_after,
        "raw_mean_signal_std": float(np.std(roi_mean_raw, ddof=0)),
        "clean_mean_signal_std": float(np.std(roi_mean_clean, ddof=0)),
    })

    for c in used_confounds:
        columns_rows.append({
            "subject": subj,
            "confound_column": c,
        })

    # -------------------------
    # E. Per-subject check figure
    # -------------------------
    max_plot_trs = min(500, roi_data_raw.shape[0])

    plt.figure(figsize=(12, 4))
    plt.plot(roi_mean_raw[:max_plot_trs], label="Raw ROI mean", linewidth=1.2)
    plt.plot(roi_mean_clean[:max_plot_trs], label="Cleaned ROI mean", linewidth=1.2)
    plt.title(f"{subj}: raw vs cleaned ROI mean timecourse")
    plt.xlabel("TR")
    plt.ylabel("Mean signal across ROI voxels")
    plt.legend(frameon=False)
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{subj}_raw_vs_cleaned_mean_timecourse.png", dpi=200)
    plt.close()

    print(f"  ROI shape (raw): {roi_data_raw.shape}")
    print(f"  Confounds used: {len(used_confounds)}")
    print(f"  Mean voxel R2 explained by confounds: {np.mean(voxel_r2):.4f}")
    print(f"  corr(ROI mean, FD):     {corr_fd_before:.4f} -> {corr_fd_after:.4f}")
    print(f"  corr(ROI mean, DVARS):  {corr_dvars_before:.4f} -> {corr_dvars_after:.4f}")
    print(f"  corr(ROI mean, Global): {corr_gsig_before:.4f} -> {corr_gsig_after:.4f}")


shape_df = pd.DataFrame(shape_rows)
shape_df.to_csv(CSV_DIR / "roi_data_shapes.csv", index=False, encoding="utf-8-sig")

confound_summary_df = pd.DataFrame(confound_rows)
confound_summary_df.to_csv(CSV_DIR / "subject_confound_summary.csv", index=False, encoding="utf-8-sig")

used_cols_df = pd.DataFrame(columns_rows)
used_cols_df.to_csv(CSV_DIR / "subject_used_confounds_long.csv", index=False, encoding="utf-8-sig")

print(f"\nExtracted RAW ROI data for {len(raw_data)} participants")
print(f"Extracted CLEANED ROI data for {len(cleaned_data)} participants")
print(f"Example participant matrix shape: {raw_data[0].shape}")


# =========================
# 11. Keep subjects with consistent voxel counts
#     Apply same filtering to raw and cleaned
# =========================

voxel_count_distribution = shape_df["n_voxels"].value_counts().sort_index()
voxel_count_distribution.to_csv(CSV_DIR / "voxel_count_distribution.csv", encoding="utf-8-sig")

print("\nVoxel count distribution across subjects:")
print(voxel_count_distribution)

common_voxel_count = int(shape_df["n_voxels"].mode().iloc[0])
print(f"\nMost common voxel count: {common_voxel_count}")

filtered_raw_data = []
filtered_cleaned_data = []
filtered_subjects = []

subject_to_raw = {subj: x for subj, x in zip(shape_df["subject"].tolist(), raw_data)}
subject_to_clean = {subj: x for subj, x in zip(shape_df["subject"].tolist(), cleaned_data)}

for subj in shape_df["subject"].tolist():
    raw_x = subject_to_raw[subj]
    clean_x = subject_to_clean[subj]

    if raw_x.shape[1] == common_voxel_count and clean_x.shape[1] == common_voxel_count:
        filtered_raw_data.append(raw_x)
        filtered_cleaned_data.append(clean_x)
        filtered_subjects.append(subj)

print(f"Subjects retained after voxel-count consistency check: {len(filtered_subjects)}")

pd.DataFrame({"subject": filtered_subjects}).to_csv(
    CSV_DIR / "subjects_retained.csv",
    index=False,
    encoding="utf-8-sig"
)

filtered_confound_summary_df = confound_summary_df[
    confound_summary_df["subject"].isin(filtered_subjects)
].copy()
filtered_confound_summary_df.to_csv(
    CSV_DIR / "subject_confound_summary_retained.csv",
    index=False,
    encoding="utf-8-sig"
)


# =========================
# 12. Step-end validation checks
# =========================

print("\n" + "=" * 80)
print("STEP-END VALIDATION CHECKS")
print("=" * 80)

if len(filtered_subjects) == 0:
    raise ValueError("No subjects retained after voxel-count consistency check.")

# Check time consistency
all_time_ok = all(x.shape[0] == STORY_N_TR for x in filtered_cleaned_data)
print(f"All retained subjects have STORY_N_TR timepoints: {all_time_ok}")
if not all_time_ok:
    raise ValueError("Timepoint inconsistency detected in cleaned data.")

# Check voxel consistency
all_vox_ok = all(x.shape[1] == common_voxel_count for x in filtered_cleaned_data)
print(f"All retained subjects have common voxel count: {all_vox_ok}")
if not all_vox_ok:
    raise ValueError("Voxel-count inconsistency detected in cleaned data.")

# Check cleaned data are finite
all_finite_ok = all(np.isfinite(x).all() for x in filtered_cleaned_data)
print(f"All cleaned arrays are finite: {all_finite_ok}")
if not all_finite_ok:
    raise ValueError("Non-finite values detected in cleaned ROI data.")

# Check grand confound reduction
grand_mean_r2 = float(filtered_confound_summary_df["mean_voxel_r2_explained_by_confounds"].mean())
grand_abs_fd_before = float(np.nanmean(np.abs(filtered_confound_summary_df["corr_roi_mean_vs_fd_before"])))
grand_abs_fd_after = float(np.nanmean(np.abs(filtered_confound_summary_df["corr_roi_mean_vs_fd_after"])))
grand_abs_dvars_before = float(np.nanmean(np.abs(filtered_confound_summary_df["corr_roi_mean_vs_dvars_before"])))
grand_abs_dvars_after = float(np.nanmean(np.abs(filtered_confound_summary_df["corr_roi_mean_vs_dvars_after"])))
grand_abs_global_before = float(np.nanmean(np.abs(filtered_confound_summary_df["corr_roi_mean_vs_global_before"])))
grand_abs_global_after = float(np.nanmean(np.abs(filtered_confound_summary_df["corr_roi_mean_vs_global_after"])))

print(f"Grand mean voxel R2 explained by confounds: {grand_mean_r2:.6f}")
print(f"Grand mean abs corr FD:     {grand_abs_fd_before:.6f} -> {grand_abs_fd_after:.6f}")
print(f"Grand mean abs corr DVARS:  {grand_abs_dvars_before:.6f} -> {grand_abs_dvars_after:.6f}")
print(f"Grand mean abs corr Global: {grand_abs_global_before:.6f} -> {grand_abs_global_after:.6f}")

if grand_abs_global_after > grand_abs_global_before:
    print("[WARNING] Global-signal correlation did not decrease on average.")
else:
    print("Check passed: global-signal coupling decreased after cleaning.")

# Quick example figure for retained subject 0
example_raw = filtered_raw_data[0]
example_clean = filtered_cleaned_data[0]

plt.figure(figsize=(10, 4))
plt.plot(np.mean(example_raw, axis=1), label="Raw", linewidth=1.2)
plt.plot(np.mean(example_clean, axis=1), label="Cleaned", linewidth=1.2)
plt.title("Example retained participant mean ROI timecourse: raw vs cleaned")
plt.xlabel("TR")
plt.ylabel("Mean signal across voxels")
plt.legend(frameon=False)
plt.tight_layout()
plt.savefig(FIG_DIR / "example_retained_raw_vs_cleaned_mean_timecourse.png", dpi=200)
plt.close()


# =========================
# 13. Save packaged arrays
#     Keep both raw and cleaned versions
# =========================

np.save(
    NPY_DIR / "roi_data_time_by_voxels_raw.npy",
    np.array(filtered_raw_data, dtype=object),
    allow_pickle=True
)

np.save(
    NPY_DIR / "roi_data_time_by_voxels_cleaned.npy",
    np.array(filtered_cleaned_data, dtype=object),
    allow_pickle=True
)

# Optional convenience alias for downstream steps if you want
np.save(
    NPY_DIR / "roi_data_time_by_voxels.npy",
    np.array(filtered_cleaned_data, dtype=object),
    allow_pickle=True
)

print(f"\nSaved RAW packaged ROI data to: {NPY_DIR / 'roi_data_time_by_voxels_raw.npy'}")
print(f"Saved CLEANED packaged ROI data to: {NPY_DIR / 'roi_data_time_by_voxels_cleaned.npy'}")
print(f"Saved downstream default ROI data to: {NPY_DIR / 'roi_data_time_by_voxels.npy'}")


# =========================
# 14. Save JSON summary/config
# =========================

config = {
    "data_dir": str(DATA_DIR),
    "project_dir": str(PROJECT_DIR),
    "run_dir": str(RUN_DIR),
    "step_dir": str(STEP_DIR),

    "atlas_path": str(ATLAS_PATH),
    "label_txt_path": str(LABEL_TXT_PATH) if LABEL_TXT_PATH.exists() else None,

    "tr": TR,
    "n_total_tr": N_TOTAL_TR,
    "story_start_tr": STORY_START_TR,
    "story_n_tr": STORY_N_TR,
    "story_end_tr": STORY_END_TR,

    "selected_parcels": SELECTED_PARCELS,
    "n_selected_parcels": len(SELECTED_PARCELS),

    "n_subjects_initial_discovered": int(len(df["subject"].unique())),
    "n_subjects_after_confound_match": int(len(manifest_df)),
    "n_subjects_retained": int(len(filtered_subjects)),

    "common_voxel_count": common_voxel_count,
    "atlas_space_roi_voxel_count": roi_voxel_count_atlas,

    "confound_cleaning": {
        "base_confound_cols": BASE_CONFOUND_COLS,
        "max_a_compcor": MAX_A_COMPCOR,
        "max_t_compcor": MAX_T_COMPCOR,
        "use_cosine_drifts": USE_COSINE_DRIFTS,
        "apply_standardize_y": APPLY_STANDARDIZE_Y,
    },

    "summary_metrics": {
        "grand_mean_voxel_r2_explained_by_confounds": grand_mean_r2,
        "grand_mean_abs_corr_fd_before": grand_abs_fd_before,
        "grand_mean_abs_corr_fd_after": grand_abs_fd_after,
        "grand_mean_abs_corr_dvars_before": grand_abs_dvars_before,
        "grand_mean_abs_corr_dvars_after": grand_abs_dvars_after,
        "grand_mean_abs_corr_global_before": grand_abs_global_before,
        "grand_mean_abs_corr_global_after": grand_abs_global_after,
    },

    "outputs": {
        "raw_roi_data_npy": str(NPY_DIR / "roi_data_time_by_voxels_raw.npy"),
        "cleaned_roi_data_npy": str(NPY_DIR / "roi_data_time_by_voxels_cleaned.npy"),
        "default_roi_data_npy": str(NPY_DIR / "roi_data_time_by_voxels.npy"),
        "subject_file_manifest_csv": str(CSV_DIR / "subject_file_manifest.csv"),
        "subject_confound_summary_csv": str(CSV_DIR / "subject_confound_summary.csv"),
        "subjects_retained_csv": str(CSV_DIR / "subjects_retained.csv"),
    },

    "run_name": RUN_NAME,
    "step_name": "step1_setup_roi_cleaned",
}

with open(STEP_DIR / "step1_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

with open(JSON_DIR / "step1_cleaned_summary.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print(f"\nSaved config JSON to: {STEP_DIR / 'step1_config.json'}")
print(f"Saved summary JSON to: {JSON_DIR / 'step1_cleaned_summary.json'}")
print("\nStep 1 (cleaned) completed successfully.")


2.0 split_zscore
Get the number of subjects and TRs
Set a train/test split ratio

Once data is loaded, we divide the data into two halves for a two fold validation. We will use one half for training SRM and the other for testing its performance. Then, we normalize the data each half.

step2_split_zscore

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import zscore


# =========================
# 1. Project setup
# =========================

PROJECT_DIR = Path(r"CHANGE_THIS_TO_YOUR_PROJECT_DIR")

# NEW cleaned run
RUN_NAME = f"{ROI_NAME}_400_parcels_cleaned"

RUN_DIR = PROJECT_DIR / "runs" / RUN_NAME
STEP1_DIR = RUN_DIR / "step1_setup_roi_cleaned"
STEP2_DIR = RUN_DIR / "step2_split_zscore"
FIG_DIR = STEP2_DIR / "figures"
CSV_DIR = STEP2_DIR / "csv"
NPY_DIR = STEP2_DIR / "npy"
JSON_DIR = STEP2_DIR / "json"

for d in [STEP2_DIR, FIG_DIR, CSV_DIR, NPY_DIR, JSON_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Project directory: {PROJECT_DIR}")
print(f"Run directory: {RUN_DIR}")
print(f"Step 1 directory: {STEP1_DIR}")
print(f"Step 2 directory: {STEP2_DIR}")


# =========================
# 2. Load Step 1 outputs
# =========================

# Keep the same downstream filename convention.
# In cleaned Step1, this file already points to the cleaned ROI data.
data_path = STEP1_DIR / "npy" / "roi_data_time_by_voxels.npy"
subjects_path = STEP1_DIR / "csv" / "subjects_retained.csv"
config_path = STEP1_DIR / "step1_config.json"

if not data_path.exists():
    raise FileNotFoundError(f"Missing Step1 ROI data: {data_path}")
if not subjects_path.exists():
    raise FileNotFoundError(f"Missing Step1 subject list: {subjects_path}")
if not config_path.exists():
    raise FileNotFoundError(f"Missing Step1 config: {config_path}")

data = np.load(data_path, allow_pickle=True)
data = [np.asarray(x, dtype=np.float32) for x in data]

subjects_df = pd.read_csv(subjects_path)
subjects = subjects_df["subject"].tolist()

with open(config_path, "r", encoding="utf-8") as f:
    step1_config = json.load(f)

print(f"Loaded ROI data from: {data_path}")
print(f"Loaded subject list from: {subjects_path}")
print(f"Loaded Step1 config from: {config_path}")
print(f"Number of retained subjects: {len(subjects)}")
print(f"Example matrix shape before split: {data[0].shape}")

if len(data) == 0:
    raise ValueError("Loaded Step1 ROI data is empty.")

if len(data) != len(subjects):
    raise ValueError(
        f"Mismatch between number of subject matrices ({len(data)}) "
        f"and subjects_retained.csv ({len(subjects)})."
    )


# =========================
# 3. Split into training and test
#    Keep same logic as original pipeline
# =========================

n_subjects = len(data)
n_trs = data[0].shape[0]

train_test_ratio = 0.5
test_size = int(n_trs * train_test_ratio)

print(f"\nn_subjects = {n_subjects}")
print(f"n_trs = {n_trs}")
print(f"train_test_ratio = {train_test_ratio}")
print(f"test_size = {test_size}")

if test_size <= 0 or test_size >= n_trs:
    raise ValueError(f"Invalid test_size={test_size} for n_trs={n_trs}")

train_data = []
test_data = []

for subject in np.arange(n_subjects):
    subj_matrix = data[subject]

    if subj_matrix.shape[0] != n_trs:
        raise ValueError(
            f"Subject index {subject} has inconsistent timepoints: "
            f"{subj_matrix.shape[0]} vs expected {n_trs}"
        )

    # First chunk = training
    train_matrix = zscore(subj_matrix[:-test_size, :], axis=0, ddof=1)
    train_matrix = np.nan_to_num(train_matrix).T.astype(np.float32)   # voxels x time
    train_data.append(train_matrix)

    # Second chunk = testing
    test_matrix = zscore(subj_matrix[-test_size:, :], axis=0, ddof=1)
    test_matrix = np.nan_to_num(test_matrix).T.astype(np.float32)     # voxels x time
    test_data.append(test_matrix)


# =========================
# 4. Inspect shapes and z-score sanity checks
# =========================

print("\nAfter split and z-scoring:")
print("Train shape (subject 1):", train_data[0].shape)
print("Test shape (subject 1):", test_data[0].shape)

print("\nSanity check on first 5 voxels of subject 1:")
print("Train mean:", np.mean(train_data[0][:5, :], axis=1))
print("Train std :", np.std(train_data[0][:5, :], axis=1, ddof=1))
print("Test mean :", np.mean(test_data[0][:5, :], axis=1))
print("Test std  :", np.std(test_data[0][:5, :], axis=1, ddof=1))

# More robust step-end checks
all_train_shapes_same = all(x.shape == train_data[0].shape for x in train_data)
all_test_shapes_same = all(x.shape == test_data[0].shape for x in test_data)
all_train_finite = all(np.isfinite(x).all() for x in train_data)
all_test_finite = all(np.isfinite(x).all() for x in test_data)

print("\nStep-end validation checks:")
print(f"All train shapes same: {all_train_shapes_same}")
print(f"All test shapes same : {all_test_shapes_same}")
print(f"All train arrays finite: {all_train_finite}")
print(f"All test arrays finite : {all_test_finite}")

if not all_train_shapes_same:
    raise ValueError("Train data shapes are inconsistent across subjects.")
if not all_test_shapes_same:
    raise ValueError("Test data shapes are inconsistent across subjects.")
if not all_train_finite:
    raise ValueError("Non-finite values found in train_data.")
if not all_test_finite:
    raise ValueError("Non-finite values found in test_data.")


# =========================
# 5. Save split summary
# =========================

split_summary = pd.DataFrame({
    "n_subjects": [n_subjects],
    "n_trs_total": [n_trs],
    "train_timepoints": [train_data[0].shape[1]],
    "test_timepoints": [test_data[0].shape[1]],
    "n_voxels": [train_data[0].shape[0]],
    "train_test_ratio": [train_test_ratio],
})

split_summary_path = CSV_DIR / "split_summary.csv"
split_summary.to_csv(split_summary_path, index=False, encoding="utf-8-sig")

print(f"\nSaved split summary to: {split_summary_path}")


# =========================
# 6. Save subject-level shapes
# =========================

shape_rows = []
for subj, train_x, test_x in zip(subjects, train_data, test_data):
    shape_rows.append({
        "subject": subj,
        "train_shape": str(train_x.shape),
        "test_shape": str(test_x.shape),
        "n_voxels": train_x.shape[0],
        "train_timepoints": train_x.shape[1],
        "test_timepoints": test_x.shape[1],
    })

shape_df = pd.DataFrame(shape_rows)
shape_df.to_csv(CSV_DIR / "subject_train_test_shapes.csv", index=False, encoding="utf-8-sig")


# =========================
# 7. Save arrays for next step
# =========================

np.save(NPY_DIR / "train_data.npy", np.array(train_data, dtype=object), allow_pickle=True)
np.save(NPY_DIR / "test_data.npy", np.array(test_data, dtype=object), allow_pickle=True)

print(f"Saved train data to: {NPY_DIR / 'train_data.npy'}")
print(f"Saved test data to: {NPY_DIR / 'test_data.npy'}")


# =========================
# 8. Optional quick visualization
# =========================

plt.figure(figsize=(10, 4))
plt.plot(np.mean(data[0], axis=1))
plt.title("Example participant mean ROI time course before split (cleaned Step1 output)")
plt.xlabel("TR")
plt.ylabel("Mean signal across voxels")
plt.tight_layout()
plt.savefig(FIG_DIR / "example_mean_roi_timecourse_before_split.png", dpi=200)
plt.close()

plt.figure(figsize=(10, 4))
plt.plot(np.mean(train_data[0], axis=0), label="Train")
plt.plot(np.mean(test_data[0], axis=0), label="Test")
plt.title("Example participant mean ROI time course after split/z-score")
plt.xlabel("TR")
plt.ylabel("Mean z-scored signal across voxels")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "example_mean_roi_timecourse_after_split.png", dpi=200)
plt.close()

print(f"Saved visualization: {FIG_DIR / 'example_mean_roi_timecourse_before_split.png'}")
print(f"Saved visualization: {FIG_DIR / 'example_mean_roi_timecourse_after_split.png'}")


# =========================
# 9. Save config
# =========================

config = {
    "n_subjects": int(n_subjects),
    "n_trs_total": int(n_trs),
    "train_test_ratio": train_test_ratio,
    "test_size": int(test_size),
    "n_voxels": int(train_data[0].shape[0]),
    "train_shape_example": list(train_data[0].shape),
    "test_shape_example": list(test_data[0].shape),
    "source_step1_config": str(config_path),
    "source_step1_dir": str(STEP1_DIR),
    "story_start_tr": int(step1_config["story_start_tr"]),
    "story_n_tr": int(step1_config["story_n_tr"]),
    "story_end_tr": int(step1_config["story_end_tr"]),
    "selected_parcels": step1_config["selected_parcels"],
    "run_name": RUN_NAME,
    "step_name": "step2_split_zscore",
    "input_roi_file": str(data_path),
    "input_is_cleaned_step1_default": True,
}

with open(STEP2_DIR / "step2_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print(f"Saved Step2 config to: {STEP2_DIR / 'step2_config.json'}")
print("\nStep 2 completed successfully.")


3.0 Estimating the SRM
Next, we train the SRM on the training data. We need to specify desired dimension of the shared feature space. Although we simply use 50 features, the optimal number of dimensions can be found using grid search with cross-validation. We also need to specify a number of iterations to ensure the SRM algorithm converges.


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from scipy.stats import zscore
import matplotlib.pyplot as plt

import brainiak.funcalign.srm
from brainiak.fcma.util import compute_correlation


# =========================
# 1. Project setup
# =========================

PROJECT_DIR = Path(r"CHANGE_THIS_TO_YOUR_PROJECT_DIR")

# NEW cleaned run
RUN_NAME = f"{ROI_NAME}_400_parcels_cleaned"

RUN_DIR = PROJECT_DIR / "runs" / RUN_NAME
STEP2_DIR = RUN_DIR / "step2_split_zscore"
STEP3a_DIR = RUN_DIR / "step3a_gridsearch_k_iter"
FIG_DIR = STEP3a_DIR / "figures"
CSV_DIR = STEP3a_DIR / "csv"
JSON_DIR = STEP3a_DIR / "json"

for d in [STEP3a_DIR, FIG_DIR, CSV_DIR, JSON_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Project directory: {PROJECT_DIR}")
print(f"Run directory: {RUN_DIR}")
print(f"Step 2 directory: {STEP2_DIR}")
print(f"Step 3a directory: {STEP3a_DIR}")


# =========================
# 2. Load train/test data
# =========================

train_path = STEP2_DIR / "npy" / "train_data.npy"
test_path = STEP2_DIR / "npy" / "test_data.npy"
step2_config_path = STEP2_DIR / "step2_config.json"

if not train_path.exists():
    raise FileNotFoundError(f"Missing train data: {train_path}")
if not test_path.exists():
    raise FileNotFoundError(f"Missing test data: {test_path}")
if not step2_config_path.exists():
    raise FileNotFoundError(f"Missing step2 config: {step2_config_path}")

train_data = np.load(train_path, allow_pickle=True)
test_data = np.load(test_path, allow_pickle=True)

train_data = [np.asarray(x, dtype=np.float32) for x in train_data]   # voxels x time
test_data = [np.asarray(x, dtype=np.float32) for x in test_data]     # voxels x time

with open(step2_config_path, "r", encoding="utf-8") as f:
    step2_config = json.load(f)

print(f"Loaded train data from: {train_path}")
print(f"Loaded test data from: {test_path}")
print(f"Loaded Step2 config from: {step2_config_path}")
print(f"Number of subjects: {len(train_data)}")
print(f"Train shape (subject 1): {train_data[0].shape}")
print(f"Test shape (subject 1): {test_data[0].shape}")

if len(train_data) == 0 or len(test_data) == 0:
    raise ValueError("Empty train_data or test_data.")

if len(train_data) != len(test_data):
    raise ValueError("train_data and test_data have different numbers of subjects.")

if not all(x.shape == train_data[0].shape for x in train_data):
    raise ValueError("Inconsistent train_data shapes across subjects.")

if not all(x.shape == test_data[0].shape for x in test_data):
    raise ValueError("Inconsistent test_data shapes across subjects.")

if not all(np.isfinite(x).all() for x in train_data):
    raise ValueError("Non-finite values found in train_data.")

if not all(np.isfinite(x).all() for x in test_data):
    raise ValueError("Non-finite values found in test_data.")


# =========================
# 3. Classification function
# =========================

def time_segment_classification(data, win_size=10):
    n_subjects = len(data)
    n_features, n_trs = data[0].shape
    accuracy = np.zeros(shape=n_subjects, dtype=np.float32)
    n_segments = n_trs - win_size + 1

    if n_segments <= 0:
        raise ValueError(
            f"win_size={win_size} is too large for n_trs={n_trs}. "
            f"Need n_trs >= win_size."
        )

    group_sum = np.zeros((n_features * win_size, n_segments), order="f", dtype=np.float32)

    for m in range(n_subjects):
        for w in range(win_size):
            group_sum[w * n_features:(w + 1) * n_features, :] += data[m][:, w:(w + n_segments)]

    for test_subject in range(n_subjects):
        subject_segments = np.zeros((n_features * win_size, n_segments), order="f", dtype=np.float32)
        for w in range(win_size):
            subject_segments[w * n_features:(w + 1) * n_features, :] = data[test_subject][:, w:(w + n_segments)]

        train_template = group_sum - subject_segments

        A = np.nan_to_num(zscore(train_template, axis=0, ddof=1))
        B = np.nan_to_num(zscore(subject_segments, axis=0, ddof=1))

        correlations = compute_correlation(B.T, A.T)

        for i in range(n_segments):
            for j in range(n_segments):
                if abs(i - j) < win_size and i != j:
                    correlations[i, j] = -np.inf

        max_idx = np.argmax(correlations, axis=1)
        accuracy[test_subject] = np.mean(max_idx == np.arange(n_segments))

    chance = 1 / np.sum(~np.isinf(correlations[n_trs // 2]))
    return accuracy, chance


# =========================
# 4. Baseline anatomical score
# =========================

WIN_SIZE = 10

acc_anat_test, chance = time_segment_classification(test_data, win_size=WIN_SIZE)

anat_mean = float(np.mean(acc_anat_test))
anat_std = float(np.std(acc_anat_test))
anat_median = float(np.median(acc_anat_test))

print("\nBaseline anatomical alignment:")
print(f"chance = {chance:.6f}")
print(f"anat_mean = {anat_mean:.6f}")
print(f"anat_median = {anat_median:.6f}")
print(f"anat_std = {anat_std:.6f}")

anat_subject_df = pd.DataFrame({
    "subject_index": np.arange(len(acc_anat_test)),
    "anat_accuracy": acc_anat_test
})
anat_subject_df.to_csv(CSV_DIR / "baseline_anatomical_subjectwise.csv", index=False, encoding="utf-8-sig")


# =========================
# 5. Grid search over K and n_iter
# =========================

K_LIST = [10, 20, 30, 50, 75, 100, 150, 200]
N_ITER_LIST = [5, 10, 20, 30]

rows = []

for n_iter in N_ITER_LIST:
    print("\n" + "#" * 90)
    print(f"Testing n_iter = {n_iter}")
    print("#" * 90)

    for k in K_LIST:
        print("\n" + "=" * 80)
        print(f"Testing K = {k}, n_iter = {n_iter}")
        print("=" * 80)

        # Safety check: K cannot exceed available voxel dimension
        if k >= train_data[0].shape[0]:
            print(f"[SKIP] K={k} >= n_voxels={train_data[0].shape[0]}")
            continue
        if k >= train_data[0].shape[1]:
            print(f"[SKIP] K={k} >= train_timepoints={train_data[0].shape[1]}")
            continue

        srm = brainiak.funcalign.srm.SRM(n_iter=n_iter, features=k)
        srm.fit(train_data)

        test_shared = srm.transform(test_data)
        test_shared = [np.nan_to_num(zscore(ts, axis=1, ddof=1)).astype(np.float32) for ts in test_shared]

        acc_shared_test, _ = time_segment_classification(test_shared, win_size=WIN_SIZE)

        shared_mean = float(np.mean(acc_shared_test))
        shared_std = float(np.std(acc_shared_test))
        shared_median = float(np.median(acc_shared_test))
        delta_mean = float(shared_mean - anat_mean)
        frac_shared_better = float(np.mean(acc_shared_test > acc_anat_test))

        rows.append({
            "k": k,
            "n_iter": n_iter,
            "win_size": WIN_SIZE,
            "chance": float(chance),

            "anat_mean": anat_mean,
            "anat_median": anat_median,
            "anat_std": anat_std,

            "shared_mean": shared_mean,
            "shared_median": shared_median,
            "shared_std": shared_std,
            "delta_mean": delta_mean,
            "subject_fraction_shared_better": frac_shared_better,
        })

        print(f"shared_mean = {shared_mean:.6f}")
        print(f"shared_median = {shared_median:.6f}")
        print(f"delta_mean  = {delta_mean:.6f}")
        print(f"fraction shared > anat = {frac_shared_better:.3f}")


# =========================
# 6. Save full results
# =========================

if len(rows) == 0:
    raise ValueError("Grid search produced no valid rows.")

results_df = pd.DataFrame(rows)
results_csv_path = CSV_DIR / "k_iter_gridsearch_results.csv"
results_df.to_csv(results_csv_path, index=False, encoding="utf-8-sig")

print(f"\nSaved full grid-search results to: {results_csv_path}")

# Best by shared_mean
best_shared_row = results_df.iloc[results_df["shared_mean"].idxmax()]

# Best by delta_mean
best_delta_row = results_df.iloc[results_df["delta_mean"].idxmax()]

summary = {
    "run_name": RUN_NAME,
    "source_step2_config": str(step2_config_path),

    "k_list": K_LIST,
    "n_iter_list": N_ITER_LIST,
    "win_size": WIN_SIZE,

    "n_subjects": int(len(train_data)),
    "train_shape_subject0": list(train_data[0].shape),
    "test_shape_subject0": list(test_data[0].shape),

    "chance": float(chance),
    "baseline_anatomical": {
        "anat_mean": anat_mean,
        "anat_median": anat_median,
        "anat_std": anat_std,
    },

    "best_by_shared_mean": {
        "k": int(best_shared_row["k"]),
        "n_iter": int(best_shared_row["n_iter"]),
        "shared_mean": float(best_shared_row["shared_mean"]),
        "shared_median": float(best_shared_row["shared_median"]),
        "shared_std": float(best_shared_row["shared_std"]),
        "delta_mean": float(best_shared_row["delta_mean"]),
        "subject_fraction_shared_better": float(best_shared_row["subject_fraction_shared_better"]),
    },

    "best_by_delta_mean": {
        "k": int(best_delta_row["k"]),
        "n_iter": int(best_delta_row["n_iter"]),
        "shared_mean": float(best_delta_row["shared_mean"]),
        "shared_median": float(best_delta_row["shared_median"]),
        "shared_std": float(best_delta_row["shared_std"]),
        "delta_mean": float(best_delta_row["delta_mean"]),
        "subject_fraction_shared_better": float(best_delta_row["subject_fraction_shared_better"]),
    },
}

with open(JSON_DIR / "step3a_gridsearch_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(f"Saved summary JSON to: {JSON_DIR / 'step3a_gridsearch_summary.json'}")


# =========================
# 7. Quick visualization
# =========================

pivot_mean = results_df.pivot(index="n_iter", columns="k", values="shared_mean")
pivot_delta = results_df.pivot(index="n_iter", columns="k", values="delta_mean")

plt.figure(figsize=(10, 5))
im = plt.imshow(pivot_mean.values, aspect="auto")
plt.xticks(np.arange(len(pivot_mean.columns)), pivot_mean.columns)
plt.yticks(np.arange(len(pivot_mean.index)), pivot_mean.index)
plt.xlabel("K")
plt.ylabel("n_iter")
plt.title("grid search: shared_mean")
plt.colorbar(im)
plt.tight_layout()
plt.savefig(FIG_DIR / "gridsearch_shared_mean.png", dpi=200)
plt.close()

plt.figure(figsize=(10, 5))
im = plt.imshow(pivot_delta.values, aspect="auto")
plt.xticks(np.arange(len(pivot_delta.columns)), pivot_delta.columns)
plt.yticks(np.arange(len(pivot_delta.index)), pivot_delta.index)
plt.xlabel("K")
plt.ylabel("n_iter")
plt.title("grid search: delta_mean (shared - anatomical)")
plt.colorbar(im)
plt.tight_layout()
plt.savefig(FIG_DIR / "gridsearch_delta_mean.png", dpi=200)
plt.close()

print(f"Saved figure: {FIG_DIR / 'gridsearch_shared_mean.png'}")
print(f"Saved figure: {FIG_DIR / 'gridsearch_delta_mean.png'}")


# =========================
# 8. Step-end validation checks
# =========================

print("\n" + "=" * 80)
print("STEP-END VALIDATION CHECKS")
print("=" * 80)

all_shared_means_finite = np.isfinite(results_df["shared_mean"]).all()
all_delta_means_finite = np.isfinite(results_df["delta_mean"]).all()
chance_valid = (chance > 0) and (chance < 1)

print(f"All shared_mean values finite: {all_shared_means_finite}")
print(f"All delta_mean values finite : {all_delta_means_finite}")
print(f"Chance in valid range (0,1): {chance_valid}")

if not all_shared_means_finite:
    raise ValueError("Non-finite shared_mean detected in grid-search results.")
if not all_delta_means_finite:
    raise ValueError("Non-finite delta_mean detected in grid-search results.")
if not chance_valid:
    raise ValueError(f"Chance value out of range: {chance}")

print("\nBest by shared_mean:")
print(pd.DataFrame([summary["best_by_shared_mean"]]).to_string(index=False))

print("\nBest by delta_mean:")
print(pd.DataFrame([summary["best_by_delta_mean"]]).to_string(index=False))

print("\nStep 3a completed successfully.")


Step3b: we use the best- K features which was selected from the previous step 3a to train the SRM then

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import brainiak.funcalign.srm


# =========================
# 1. Project setup
# =========================

PROJECT_DIR = Path(r"CHANGE_THIS_TO_YOUR_PROJECT_DIR")

# NEW cleaned run
RUN_NAME = f"{ROI_NAME}_400_parcels_cleaned"

RUN_DIR = PROJECT_DIR / "runs" / RUN_NAME
STEP2_DIR = RUN_DIR / "step2_split_zscore"
STEP3a_DIR = RUN_DIR / "step3a_gridsearch_k_iter"
STEP3b_DIR = RUN_DIR / "step3b_estimate_srm"
FIG_DIR = STEP3b_DIR / "figures"
CSV_DIR = STEP3b_DIR / "csv"
NPY_DIR = STEP3b_DIR / "npy"
JSON_DIR = STEP3b_DIR / "json"

for d in [STEP3b_DIR, FIG_DIR, CSV_DIR, NPY_DIR, JSON_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Project directory: {PROJECT_DIR}")
print(f"Run directory: {RUN_DIR}")
print(f"Step 2 directory: {STEP2_DIR}")
print(f"Step 3a directory: {STEP3a_DIR}")
print(f"Step 3b directory: {STEP3b_DIR}")


# =========================
# 2. Load train/test data
# =========================

train_path = STEP2_DIR / "npy" / "train_data.npy"
test_path = STEP2_DIR / "npy" / "test_data.npy"
step2_config_path = STEP2_DIR / "step2_config.json"
step3a_summary_path = STEP3a_DIR / "json" / "step3a_gridsearch_summary.json"

if not train_path.exists():
    raise FileNotFoundError(f"Missing train data: {train_path}")
if not test_path.exists():
    raise FileNotFoundError(f"Missing test data: {test_path}")
if not step2_config_path.exists():
    raise FileNotFoundError(f"Missing step2 config: {step2_config_path}")

train_data = np.load(train_path, allow_pickle=True)
test_data = np.load(test_path, allow_pickle=True)

train_data = [np.asarray(x, dtype=np.float32) for x in train_data]
test_data = [np.asarray(x, dtype=np.float32) for x in test_data]

with open(step2_config_path, "r", encoding="utf-8") as f:
    step2_config = json.load(f)

print(f"Loaded train data from: {train_path}")
print(f"Loaded test data from: {test_path}")
print(f"Number of subjects: {len(train_data)}")
print(f"Train shape (subject 1): {train_data[0].shape}")
print(f"Test shape (subject 1): {test_data[0].shape}")

if len(train_data) == 0 or len(test_data) == 0:
    raise ValueError("Empty train_data or test_data.")
if len(train_data) != len(test_data):
    raise ValueError("train_data and test_data have different numbers of subjects.")
if not all(x.shape == train_data[0].shape for x in train_data):
    raise ValueError("Inconsistent train_data shapes across subjects.")
if not all(x.shape == test_data[0].shape for x in test_data):
    raise ValueError("Inconsistent test_data shapes across subjects.")
if not all(np.isfinite(x).all() for x in train_data):
    raise ValueError("Non-finite values found in train_data.")
if not all(np.isfinite(x).all() for x in test_data):
    raise ValueError("Non-finite values found in test_data.")


# =========================
# 3. Choose final SRM parameters
#    Default: use best grid-search result
# =========================

if step3a_summary_path.exists():
    with open(step3a_summary_path, "r", encoding="utf-8") as f:
        step3a_summary = json.load(f)

    FEATURES = int(step3a_summary["best_by_shared_mean"]["k"])
    N_ITER = int(step3a_summary["best_by_shared_mean"]["n_iter"])

    print("\nLoaded best parameters from Step3a summary:")
    print(f"FEATURES = {FEATURES}")
    print(f"N_ITER   = {N_ITER}")
else:
    # Fallback to the values you just found
    FEATURES = 30
    N_ITER = 20

    print("\nStep3a summary not found. Using fallback parameters:")
    print(f"FEATURES = {FEATURES}")
    print(f"N_ITER   = {N_ITER}")

if FEATURES >= train_data[0].shape[0]:
    raise ValueError(f"FEATURES={FEATURES} must be < n_voxels={train_data[0].shape[0]}")
if FEATURES >= train_data[0].shape[1]:
    raise ValueError(f"FEATURES={FEATURES} must be < train_timepoints={train_data[0].shape[1]}")


# =========================
# 4. Estimate SRM on training data
# =========================

srm = brainiak.funcalign.srm.SRM(
    n_iter=N_ITER,
    features=FEATURES
)

print("\nFitting SRM on cleaned training data...")
srm.fit(train_data)
print("SRM has been fit successfully.")

print(f"Shared response shape: {srm.s_.shape[0]} features x {srm.s_.shape[1]} time-points")
print(f"Number of subject-specific weight matrices: {len(srm.w_)}")
print(f"Weight matrix shape (subject 0): {srm.w_[0].shape}")


# =========================
# 5. Orthogonality check of weight matrix
# =========================

subject = 0

orthogonality_matrix = srm.w_[subject].T.dot(srm.w_[subject])

sns.set_style("white")
fig, ax = plt.subplots(1, figsize=(6, 5))
m = ax.matshow(orthogonality_matrix)
ax.set_title(f"Weight matrix orthogonality for subject {subject}", pad=10)
ax.set_xlabel("SRM features")
ax.set_ylabel("SRM features")
ax.tick_params(length=0)
cbar = fig.colorbar(m, ax=ax, ticks=[0, 1])
cbar.ax.tick_params(length=0)
plt.tight_layout()
plt.savefig(FIG_DIR / f"weight_matrix_orthogonality_subject_{subject}.png", dpi=200)
plt.close()

print(f"\nWeight matrix shape: {srm.w_[subject].shape[0]} voxels x {srm.w_[subject].shape[1]} features")

if np.allclose(np.identity(FEATURES), orthogonality_matrix):
    orthogonal_check = True
    print(f"This confirms that the weight matrix for subject {subject} is orthogonal (within tolerance).")
else:
    orthogonal_check = False
    print("Weight matrix is not perfectly orthogonal under strict comparison.")
    print("This can still happen numerically; allclose tolerance is the practical criterion.")


# =========================
# 6. Transform training and test data
# =========================

print("\nTransforming train and test data into shared space...")

shared_train = srm.transform(train_data)
shared_test = srm.transform(test_data)

shared_train = [np.asarray(x, dtype=np.float32) for x in shared_train]
shared_test = [np.asarray(x, dtype=np.float32) for x in shared_test]

print(f"Shared train shape (subject 1): {shared_train[0].shape}")
print(f"Shared test shape (subject 1): {shared_test[0].shape}")


# =========================
# 7. Save outputs
# =========================

# Save the model summary tables
pd.DataFrame(srm.s_).to_csv(CSV_DIR / "shared_response.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(orthogonality_matrix).to_csv(
    CSV_DIR / f"weight_matrix_orthogonality_subject_{subject}.csv",
    index=False,
    encoding="utf-8-sig"
)

summary_df = pd.DataFrame({
    "n_subjects": [len(train_data)],
    "n_voxels": [train_data[0].shape[0]],
    "train_timepoints": [train_data[0].shape[1]],
    "test_timepoints": [test_data[0].shape[1]],
    "features": [FEATURES],
    "n_iter": [N_ITER],
    "shared_shape": [str(srm.s_.shape)],
    "weight_shape_subject0": [str(srm.w_[subject].shape)],
    "orthogonal_check": [bool(orthogonal_check)],
})
summary_df.to_csv(CSV_DIR / "srm_fit_summary.csv", index=False, encoding="utf-8-sig")

# Save npy outputs
np.save(NPY_DIR / "shared_response.npy", srm.s_)
np.save(NPY_DIR / "weight_matrices.npy", np.array(srm.w_, dtype=object), allow_pickle=True)
np.save(NPY_DIR / "shared_train.npy", np.array(shared_train, dtype=object), allow_pickle=True)
np.save(NPY_DIR / "shared_test.npy", np.array(shared_test, dtype=object), allow_pickle=True)

config = {
    "features": FEATURES,
    "n_iter": N_ITER,
    "n_subjects": int(len(train_data)),
    "n_voxels": int(train_data[0].shape[0]),
    "train_timepoints": int(train_data[0].shape[1]),
    "test_timepoints": int(test_data[0].shape[1]),
    "shared_shape": list(srm.s_.shape),
    "weight_shape_subject0": list(srm.w_[subject].shape),
    "orthogonal_check": bool(orthogonal_check),
    "run_name": RUN_NAME,
    "step_name": "step3b_estimate_srm",
    "source_step2_config": str(step2_config_path),
    "source_step3a_summary": str(step3a_summary_path) if step3a_summary_path.exists() else None,
}
with open(STEP3b_DIR / "step3b_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

with open(JSON_DIR / "step3b_summary.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)


# =========================
# 8. Step-end validation checks
# =========================

print("\n" + "=" * 80)
print("STEP-END VALIDATION CHECKS")
print("=" * 80)

shared_train_shapes_same = all(x.shape == shared_train[0].shape for x in shared_train)
shared_test_shapes_same = all(x.shape == shared_test[0].shape for x in shared_test)
shared_train_finite = all(np.isfinite(x).all() for x in shared_train)
shared_test_finite = all(np.isfinite(x).all() for x in shared_test)
weight_shapes_same = all(w.shape == srm.w_[0].shape for w in srm.w_)

print(f"All shared_train shapes same: {shared_train_shapes_same}")
print(f"All shared_test shapes same : {shared_test_shapes_same}")
print(f"All shared_train finite    : {shared_train_finite}")
print(f"All shared_test finite     : {shared_test_finite}")
print(f"All weight matrix shapes same: {weight_shapes_same}")
print(f"Shared response shape      : {srm.s_.shape}")
print(f"Orthogonal check (subject 0): {orthogonal_check}")

if not shared_train_shapes_same:
    raise ValueError("Inconsistent shared_train shapes across subjects.")
if not shared_test_shapes_same:
    raise ValueError("Inconsistent shared_test shapes across subjects.")
if not shared_train_finite:
    raise ValueError("Non-finite values found in shared_train.")
if not shared_test_finite:
    raise ValueError("Non-finite values found in shared_test.")
if not weight_shapes_same:
    raise ValueError("Inconsistent SRM weight matrix shapes across subjects.")

print("\nSaved outputs:")
print(f"Shared response CSV: {CSV_DIR / 'shared_response.csv'}")
print(f"Shared response NPY: {NPY_DIR / 'shared_response.npy'}")
print(f"Weight matrices NPY: {NPY_DIR / 'weight_matrices.npy'}")
print(f"Shared train NPY: {NPY_DIR / 'shared_train.npy'}")
print(f"Shared test NPY: {NPY_DIR / 'shared_test.npy'}")
print("\nStep 3b completed successfully.")


We confirmed that each subject-specific SRM loading matrix was approximately orthonormal (𝑊𝑖⊤𝑊𝑖≈𝐼); see  Figure

This figure shows 𝑊⊤𝑊 for one subject’s SRM weight matrix. The bright diagonal and near-zero off-diagonal values indicate that the learned SRM features are approximately orthonormal, as expected.

The plot verifies that the subject-specific SRM weight matrix satisfies the orthogonality constraint. Each latent feature is normalized, and different features are nearly orthogonal to one another, meaning the shared dimensions are not redundant.

4.0 Between-subject time-segment classification
When we trained SRM above, we learned the weight matrices 
Wi and the shared response S for the training data. The weight matrices further allow us to convert new data to the shared feature space. We call the transform() function to transform test data for each subject into the shred space.

step4_time_segment_classification_21styear

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import zscore
from brainiak.fcma.util import compute_correlation


# =========================
# 1. Project setup
# =========================

PROJECT_DIR = Path(r"CHANGE_THIS_TO_YOUR_PROJECT_DIR")

# NEW cleaned run
RUN_NAME = f"{ROI_NAME}_400_parcels_cleaned"

RUN_DIR = PROJECT_DIR / "runs" / RUN_NAME
STEP2_DIR = RUN_DIR / "step2_split_zscore"
STEP3b_DIR = RUN_DIR / "step3b_estimate_srm"
STEP4_DIR = RUN_DIR / "step4_time_segment_classification"

FIG_DIR = STEP4_DIR / "figures"
CSV_DIR = STEP4_DIR / "csv"
NPY_DIR = STEP4_DIR / "npy"
JSON_DIR = STEP4_DIR / "json"

for d in [STEP4_DIR, FIG_DIR, CSV_DIR, NPY_DIR, JSON_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Project directory: {PROJECT_DIR}")
print(f"Run directory: {RUN_DIR}")
print(f"Step 2 directory: {STEP2_DIR}")
print(f"Step 3b directory: {STEP3b_DIR}")
print(f"Step 4 directory: {STEP4_DIR}")


# =========================
# 2. Load train/test data
# =========================

train_path = STEP2_DIR / "npy" / "train_data.npy"
test_path = STEP2_DIR / "npy" / "test_data.npy"
step2_config_path = STEP2_DIR / "step2_config.json"

if not train_path.exists():
    raise FileNotFoundError(f"Missing train data: {train_path}")
if not test_path.exists():
    raise FileNotFoundError(f"Missing test data: {test_path}")
if not step2_config_path.exists():
    raise FileNotFoundError(f"Missing Step2 config: {step2_config_path}")

train_data = np.load(train_path, allow_pickle=True)
test_data = np.load(test_path, allow_pickle=True)

train_data = [np.asarray(x, dtype=np.float32) for x in train_data]
test_data = [np.asarray(x, dtype=np.float32) for x in test_data]

with open(step2_config_path, "r", encoding="utf-8") as f:
    step2_config = json.load(f)

print(f"Loaded train data from: {train_path}")
print(f"Loaded test data from: {test_path}")
print(f"Number of subjects: {len(train_data)}")
print(f"Train shape (subject 1): {train_data[0].shape}")
print(f"Test shape (subject 1): {test_data[0].shape}")

if len(train_data) == 0 or len(test_data) == 0:
    raise ValueError("Empty train_data or test_data.")
if len(train_data) != len(test_data):
    raise ValueError("Mismatch between train_data and test_data subject counts.")
if not all(x.shape == train_data[0].shape for x in train_data):
    raise ValueError("Inconsistent train_data shapes across subjects.")
if not all(x.shape == test_data[0].shape for x in test_data):
    raise ValueError("Inconsistent test_data shapes across subjects.")
if not all(np.isfinite(x).all() for x in train_data):
    raise ValueError("Non-finite values found in train_data.")
if not all(np.isfinite(x).all() for x in test_data):
    raise ValueError("Non-finite values found in test_data.")


# =========================
# 3. Load SRM outputs from Step3b
# =========================

w_path = STEP3b_DIR / "npy" / "weight_matrices.npy"
config_path = STEP3b_DIR / "step3b_config.json"

if not w_path.exists():
    raise FileNotFoundError(
        f"Could not find SRM W matrices from Step3b: {w_path}\n"
        "Please make sure Step3b saved np.array(srm.w_, dtype=object) to this path."
    )
if not config_path.exists():
    raise FileNotFoundError(f"Missing Step3b config: {config_path}")

W_list = np.load(w_path, allow_pickle=True)
W_list = [np.asarray(w, dtype=np.float32) for w in W_list]

with open(config_path, "r", encoding="utf-8") as f:
    step3b_config = json.load(f)

print(f"\nLoaded W matrices from: {w_path}")
print(f"Number of W matrices: {len(W_list)}")
print(f"W shape (subject 1): {W_list[0].shape}")

if len(W_list) != len(test_data):
    raise ValueError(
        f"Mismatch between number of W matrices ({len(W_list)}) "
        f"and number of test subjects ({len(test_data)})."
    )

if not all(w.shape == W_list[0].shape for w in W_list):
    raise ValueError("Inconsistent W matrix shapes across subjects.")

if not all(np.isfinite(w).all() for w in W_list):
    raise ValueError("Non-finite values found in W matrices.")

FEATURES = int(step3b_config["features"])
print(f"Recovered FEATURES from Step3b: {FEATURES}")


# =========================
# 4. Time-segment classification function
# =========================

def time_segment_classification(data, win_size=10):
    """
    data: list of subject matrices, each [features, time]
    """
    n_subjects = len(data)
    n_features, n_trs = data[0].shape
    accuracy = np.zeros(shape=n_subjects, dtype=np.float32)
    n_segments = n_trs - win_size + 1

    if n_segments <= 0:
        raise ValueError(
            f"win_size={win_size} is too large for n_trs={n_trs}. "
            f"Need n_trs >= win_size."
        )

    group_sum = np.zeros((n_features * win_size, n_segments), order="f", dtype=np.float32)

    for m in range(n_subjects):
        for w in range(win_size):
            group_sum[w * n_features:(w + 1) * n_features, :] += data[m][:, w:(w + n_segments)]

    for test_subject in range(n_subjects):
        subject_segments = np.zeros((n_features * win_size, n_segments), order="f", dtype=np.float32)
        for w in range(win_size):
            subject_segments[w * n_features:(w + 1) * n_features, :] = data[test_subject][:, w:(w + n_segments)]

        train_template = group_sum - subject_segments

        A = np.nan_to_num(zscore(train_template, axis=0, ddof=1))
        B = np.nan_to_num(zscore(subject_segments, axis=0, ddof=1))

        correlations = compute_correlation(B.T, A.T)

        for i in range(n_segments):
            for j in range(n_segments):
                if abs(i - j) < win_size and i != j:
                    correlations[i, j] = -np.inf

        max_idx = np.argmax(correlations, axis=1)
        accuracy[test_subject] = np.mean(max_idx == np.arange(n_segments))

    chance = 1 / np.sum(~np.isinf(correlations[n_trs // 2]))
    return accuracy, chance


# =========================
# 5. Baseline anatomical alignment
# =========================

WIN_SIZE = 10

acc_anat_test, chance = time_segment_classification(test_data, win_size=WIN_SIZE)

anat_mean = float(np.mean(acc_anat_test))
anat_median = float(np.median(acc_anat_test))
anat_std = float(np.std(acc_anat_test))

print("\nBaseline anatomical alignment:")
print(f"chance = {chance:.6f}")
print(f"anat_mean = {anat_mean:.6f}")
print(f"anat_median = {anat_median:.6f}")
print(f"anat_std = {anat_std:.6f}")


# =========================
# 6. Project test data into shared space using W
# =========================

shared_test = []

for subj_idx, (X_test_subj, W_subj) in enumerate(zip(test_data, W_list)):
    # X_test_subj: [voxels, time]
    # W_subj:      [voxels, features]
    # shared:      [features, time]
    X_shared = W_subj.T.dot(X_test_subj).astype(np.float32)

    # z-score each shared feature across time
    X_shared = np.nan_to_num(zscore(X_shared, axis=1, ddof=1)).astype(np.float32)

    shared_test.append(X_shared)

print("\nProjected test data into shared space.")
print(f"Shared test shape (subject 1): {shared_test[0].shape}")


# =========================
# 7. Shared-space time-segment classification
# =========================

acc_shared_test, _ = time_segment_classification(shared_test, win_size=WIN_SIZE)

shared_mean = float(np.mean(acc_shared_test))
shared_median = float(np.median(acc_shared_test))
shared_std = float(np.std(acc_shared_test))
delta_mean = float(shared_mean - anat_mean)
subject_fraction_shared_better = float(np.mean(acc_shared_test > acc_anat_test))

print("\nShared-space alignment:")
print(f"shared_mean = {shared_mean:.6f}")
print(f"shared_median = {shared_median:.6f}")
print(f"shared_std = {shared_std:.6f}")
print(f"delta_mean = {delta_mean:.6f}")
print(f"subject_fraction_shared_better = {subject_fraction_shared_better:.6f}")


# =========================
# 8. Save subjectwise results
# =========================

subject_df = pd.DataFrame({
    "subject_index": np.arange(len(acc_anat_test)),
    "anat_accuracy": acc_anat_test,
    "shared_accuracy": acc_shared_test,
    "shared_minus_anat": acc_shared_test - acc_anat_test,
    "shared_better": acc_shared_test > acc_anat_test,
})

subject_csv_path = CSV_DIR / "subjectwise_time_segment_classification.csv"
subject_df.to_csv(subject_csv_path, index=False, encoding="utf-8-sig")

print(f"\nSaved subjectwise results to: {subject_csv_path}")


# =========================
# 9. Save summary table
# =========================

summary_df = pd.DataFrame({
    "n_subjects": [len(test_data)],
    "win_size": [WIN_SIZE],
    "chance": [chance],

    "anat_mean": [anat_mean],
    "anat_median": [anat_median],
    "anat_std": [anat_std],

    "shared_mean": [shared_mean],
    "shared_median": [shared_median],
    "shared_std": [shared_std],

    "delta_mean": [delta_mean],
    "subject_fraction_shared_better": [subject_fraction_shared_better],
})

summary_csv_path = CSV_DIR / "step4_summary.csv"
summary_df.to_csv(summary_csv_path, index=False, encoding="utf-8-sig")

print(f"Saved summary table to: {summary_csv_path}")


# =========================
# 10. Save arrays
# =========================

np.save(NPY_DIR / "acc_anat_test.npy", acc_anat_test)
np.save(NPY_DIR / "acc_shared_test.npy", acc_shared_test)
np.save(NPY_DIR / "shared_test_projected.npy", np.array(shared_test, dtype=object), allow_pickle=True)

print(f"Saved anatomical accuracy array to: {NPY_DIR / 'acc_anat_test.npy'}")
print(f"Saved shared-space accuracy array to: {NPY_DIR / 'acc_shared_test.npy'}")
print(f"Saved projected shared test data to: {NPY_DIR / 'shared_test_projected.npy'}")


# =========================
# 11. Visualization
# =========================

# (A) Scatter plot
plt.figure(figsize=(8, 5))
plt.scatter(acc_anat_test, acc_shared_test, alpha=0.8)
min_val = min(np.min(acc_anat_test), np.min(acc_shared_test))
max_val = max(np.max(acc_anat_test), np.max(acc_shared_test))
plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")
plt.xlabel("Anatomical test accuracy")
plt.ylabel("Shared-space test accuracy")
plt.title("Subjectwise time-segment classification")
plt.tight_layout()
plt.savefig(FIG_DIR / "subjectwise_anatomical_vs_shared.png", dpi=200)
plt.close()

# (B) Bar plot of improvement
plt.figure(figsize=(10, 4))
plt.bar(np.arange(len(acc_anat_test)), acc_shared_test - acc_anat_test)
plt.axhline(0, linestyle="--")
plt.xlabel("Subject index")
plt.ylabel("Shared - anatomical")
plt.title("Subjectwise improvement in time-segment classification")
plt.tight_layout()
plt.savefig(FIG_DIR / "subjectwise_shared_minus_anatomical.png", dpi=200)
plt.close()

# (C) OLD-STYLE BOXPLOT (kept from your previous version)
plot_df = pd.DataFrame({
    "classification accuracy": np.concatenate([acc_anat_test, acc_shared_test]),
    "alignment": (["anatomical\nalignment"] * len(acc_anat_test)) + (["SRM"] * len(acc_shared_test))
})

plt.figure(figsize=(8, 5))
plt.boxplot(
    [acc_anat_test, acc_shared_test],
    labels=["anatomical\nalignment", "SRM"],
    widths=0.35
)
plt.axhline(chance, linestyle="--", color="gray")
plt.ylabel("classification accuracy")
plt.xlabel("alignment")
plt.title("Between-subject time-segment classification")
plt.tight_layout()
plt.savefig(FIG_DIR / "between_subject_time_segment_classification_boxplot.png", dpi=200)
plt.close()

print(f"Saved figure: {FIG_DIR / 'subjectwise_anatomical_vs_shared.png'}")
print(f"Saved figure: {FIG_DIR / 'subjectwise_shared_minus_anatomical.png'}")
print(f"Saved figure: {FIG_DIR / 'between_subject_time_segment_classification_boxplot.png'}")


# =========================
# 12. Save config JSON
# =========================

config = {
    "run_name": RUN_NAME,
    "step_name": "step4_time_segment_classification",

    "source_step2_config": str(step2_config_path),
    "source_step3b_config": str(config_path),

    "n_subjects": int(len(test_data)),
    "train_shape_subject0": list(train_data[0].shape),
    "test_shape_subject0": list(test_data[0].shape),
    "w_shape_subject0": list(W_list[0].shape),
    "shared_test_shape_subject0": list(shared_test[0].shape),

    "features": FEATURES,
    "win_size": WIN_SIZE,
    "chance": float(chance),

    "anat_mean": anat_mean,
    "anat_median": anat_median,
    "anat_std": anat_std,

    "shared_mean": shared_mean,
    "shared_median": shared_median,
    "shared_std": shared_std,

    "delta_mean": delta_mean,
    "subject_fraction_shared_better": subject_fraction_shared_better,
}

with open(STEP4_DIR / "step4_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

with open(JSON_DIR / "step4_summary.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print(f"Saved Step4 config to: {STEP4_DIR / 'step4_config.json'}")


# =========================
# 13. Step-end validation checks
# =========================

print("\n" + "=" * 80)
print("STEP-END VALIDATION CHECKS")
print("=" * 80)

shared_test_shapes_same = all(x.shape == shared_test[0].shape for x in shared_test)
shared_test_finite = all(np.isfinite(x).all() for x in shared_test)
acc_anat_finite = np.isfinite(acc_anat_test).all()
acc_shared_finite = np.isfinite(acc_shared_test).all()
chance_valid = (chance > 0) and (chance < 1)

print(f"All shared_test shapes same: {shared_test_shapes_same}")
print(f"All shared_test finite    : {shared_test_finite}")
print(f"Anatomical accuracies finite: {acc_anat_finite}")
print(f"Shared accuracies finite    : {acc_shared_finite}")
print(f"Chance in valid range (0,1): {chance_valid}")

if not shared_test_shapes_same:
    raise ValueError("Inconsistent shared_test shapes across subjects.")
if not shared_test_finite:
    raise ValueError("Non-finite values found in shared_test.")
if not acc_anat_finite:
    raise ValueError("Non-finite values found in acc_anat_test.")
if not acc_shared_finite:
    raise ValueError("Non-finite values found in acc_shared_test.")
if not chance_valid:
    raise ValueError(f"Chance value out of valid range: {chance}")

print("\nFinal Step4 summary:")
print(summary_df.to_string(index=False))

print("\nStep 4 completed successfully.")
